In [1]:
import os
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# Function to load results from folders, including two_step_model
def load_results(base_path):
    data = []
    # Add the new folder name "two_step_model" to the model types
    for model_type in ["random_forest_binary", "random_forest_regression", "xgboost_binary", "xgboost_regression", "two_step_model"]:
        model_path = os.path.join(base_path, model_type)
        if not os.path.exists(model_path):
            continue
        for seed_folder in os.listdir(model_path):
            seed_path = os.path.join(model_path, seed_folder)
            # Skip _Duplicate folders
            if "_Duplicate" in seed_folder or not os.path.isdir(seed_path):
                continue
            overall_results_path = os.path.join(seed_path, "overall_results.json")
            if os.path.exists(overall_results_path):
                with open(overall_results_path, "r") as file:
                    results = json.load(file)
                method = "lf" if seed_folder.endswith("_lf") else "extent"
                data.append({
                    "method": method,
                    "model_type": model_type,
                    "seed": seed_folder,
                    "results": results
                })
    return data

# Function to aggregate metrics across seeds, including two_step_model
def aggregate_metrics(data):
    aggregated_data = []
    for method in ["extent", "lf"]:
        for model_type in ["random_forest_binary", "random_forest_regression", "xgboost_binary", "xgboost_regression", "two_step_model"]:
            relevant_results = [d["results"]["metrics"]["weighted_average"] for d in data if d["method"] == method and d["model_type"] == model_type]
            if not relevant_results:
                continue
            # Dynamically extract all metrics found in the first result
            first_result = relevant_results[0]
            metrics = first_result.keys()
            for metric in metrics:
                # Collect the "metric" value for each metric across all relevant results
                values = [result.get(metric, {}).get("metric", None) for result in relevant_results]
                values = [v for v in values if v is not None]  # Filter out None values
                if values:
                    aggregated_data.append({
                        "method": method,
                        "model_type": model_type,
                        "metric": metric,
                        "mean": round(sum(values) / len(values), 4),
                        "std": round(pd.Series(values).std(), 4),
                        "all_values": values  # Store all the values for later viewing
                    })
    return pd.DataFrame(aggregated_data)

# Load data from the specified base directory
base_path = "results_per_model"
data = load_results(base_path)

# Aggregate metrics across seeds
aggregated_df = aggregate_metrics(data)

# Function to update the metrics dropdown based on selected model type
def update_metrics_dropdown(model_type):
    if "binary" in model_type:
        metrics = sorted(aggregated_df[aggregated_df["model_type"].str.contains("binary")]["metric"].unique())
    elif "regression" in model_type:
        metrics = sorted(aggregated_df[aggregated_df["model_type"].str.contains("regression")]["metric"].unique())
    else:
        metrics = sorted(aggregated_df[aggregated_df["model_type"] == "two_step_model"]["metric"].unique())
    metrics_dropdown.options = [(metric, metric) for metric in metrics]
    metrics_dropdown.value = metrics  # Select all by default

# Function to update the table display
def update_table(selected_model, selected_metrics, show_all_values=False):
    filtered_df = aggregated_df[
        (aggregated_df["model_type"].str.contains(selected_model)) &
        (aggregated_df["metric"].isin(selected_metrics))
    ]

    display_columns = ["method", "model_type", "metric", "mean", "std"]
    if show_all_values:
        display_columns.append("all_values")
    
    display_df = filtered_df[display_columns]

    # Convert DataFrame to HTML table
    table_html = display_df.to_html(index=False, justify='left')
    
    # Display the table
    display(HTML(table_html))

# Create Dropdown Widgets
model_type_dropdown = widgets.Dropdown(
    options=[
        ("Binary Models", "binary"),
        ("Regression Models", "regression"),
        ("Two Step Model", "two_step_model")
    ],
    value="binary",
    description="Model Type:"
)

metrics_dropdown = widgets.SelectMultiple(
    options=[],  # This will be populated based on the model type selected
    description="Metrics:"
)

# Update metrics dropdown when the model type changes
model_type_dropdown.observe(lambda change: update_metrics_dropdown(change.new), names='value')

# Create Checkbox Widget to toggle showing all values
show_all_values_checkbox = widgets.Checkbox(
    value=False,
    description="Show all values",
    disabled=False
)

# Interactive output
interactive_output = widgets.interactive_output(update_table, {
    'selected_model': model_type_dropdown,
    'selected_metrics': metrics_dropdown,
    'show_all_values': show_all_values_checkbox
})

# Initial update of metrics dropdown
update_metrics_dropdown(model_type_dropdown.value)

# Display the dropdowns, checkbox, and table
display(model_type_dropdown)
display(metrics_dropdown)
display(show_all_values_checkbox)
display(interactive_output)


Dropdown(description='Model Type:', options=(('Binary Models', 'binary'), ('Regression Models', 'regression'),…

SelectMultiple(description='Metrics:', index=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16), optio…

Checkbox(value=False, description='Show all values')

Output()

In [7]:
import os
import json
import pandas as pd
from IPython.display import display, HTML

# Configuration
BASE_PATH = "results_per_model"
TWO_STEP_PATH = "results_per_model/two_step_model"  # Path to the two_step_model folder
MODEL_TYPES = [
    "random_forest_binary",
    "xgboost_binary",
    "random_forest_regression",
    "xgboost_regression"
]
RAINFALL_METHODS = ["ext", "lf"]

# Define the maximum number of splits per seed
MAX_SPLIT_COUNT_PER_SEED = 5

FEATURE_COMBINATIONS = {
    "during_total_rainfall": {  # This is the "EXT" feature key
        "lf_counterpart": "lf_during_max_6h_intensity",
        "combined_name": "during_total_rainfall / lf_during_max_6h_intensity"
    },
    "post_max_6h": {
        "lf_counterpart": "lf_post_max_24h",
        "combined_name": "post_max_6h / lf_post_max_24h"
    },
    "pre_max_6h": {
        "lf_counterpart": "lf_pre_max_6h",
        "combined_name": "pre_max_6h / lf_pre_max_6h"
    }
}


def load_model_results():
    results = {}

    for model_type_name in MODEL_TYPES:
        base_model_prefix = ""
        model_type_suffix = ""

        if model_type_name.startswith("random_forest"):
            base_model_prefix = "RF"
            if "binary" in model_type_name:
                model_type_suffix = "Binary"
            elif "regression" in model_type_name:
                model_type_suffix = "Regression"
            else:
                model_type_suffix = model_type_name.split('_')[-1].capitalize()
        elif model_type_name.startswith("xgboost"):
            base_model_prefix = "XGB"
            if "binary" in model_type_name:
                model_type_suffix = "Binary"
            elif "regression" in model_type_name:
                model_type_suffix = "Regression"
            else:
                model_type_suffix = model_type_name.split('_')[-1].capitalize()
        else:
            base_model_prefix = model_type_name.split('_')[0].upper()
            model_type_suffix = model_type_name.split('_')[-1].capitalize()

        for method_name in RAINFALL_METHODS:
            col_name = f"{base_model_prefix} {model_type_suffix} {method_name.upper()}"
            results[col_name] = {}

    for model in MODEL_TYPES:
        variant_path = os.path.join(BASE_PATH, model)

        if not os.path.exists(variant_path):
            continue

        base_model_prefix = ""
        model_type_suffix = ""

        if model.startswith("random_forest"):
            base_model_prefix = "RF"
            if "binary" in model:
                model_type_suffix = "Binary"
            elif "regression" in model:
                model_type_suffix = "Regression"
            else:
                model_type_suffix = model.split('_')[-1].capitalize()
        elif model.startswith("xgboost"):
            base_model_prefix = "XGB"
            if "binary" in model:
                model_type_suffix = "Binary"
            elif "regression" in model:
                model_type_suffix = "Regression"
            else:
                model_type_suffix = model.split('_')[-1].capitalize()
        else:
            base_model_prefix = model.split('_')[0].upper()
            model_type_suffix = model.split('_')[-1].capitalize()

        feature_total_split_selections_ext = {}
        feature_total_split_selections_lf = {}
        num_ext_seeds = 0
        num_lf_seeds = 0

        for seed_folder in os.listdir(variant_path):
            seed_folder_full_path = os.path.join(variant_path, seed_folder)
            if "_Duplicate" in seed_folder or not os.path.isdir(seed_folder_full_path):
                continue

            results_path = os.path.join(seed_folder_full_path, "overall_results.json")
            if os.path.exists(results_path):
                try:
                    with open(results_path, "r") as f:
                        data = json.load(f)

                    if "selected_features_count" in data and data["selected_features_count"]:
                        if "_lf" in seed_folder.lower():
                            target_feature_totals = feature_total_split_selections_lf
                            num_lf_seeds += 1
                        else:
                            target_feature_totals = feature_total_split_selections_ext
                            num_ext_seeds += 1

                        for feature, count_in_splits in data["selected_features_count"].items():
                            target_feature_totals[feature] = target_feature_totals.get(feature, 0) + count_in_splits

                except (json.JSONDecodeError, Exception):
                    pass

        ext_col_name = f"{base_model_prefix} {model_type_suffix} EXT"
        total_possible_ext_selections = num_ext_seeds * MAX_SPLIT_COUNT_PER_SEED
        for feature, total_selections in feature_total_split_selections_ext.items():
            if total_possible_ext_selections > 0:
                percentage = (total_selections / total_possible_ext_selections) * 100
                results[ext_col_name][feature] = f"{percentage:.0f}%"
            else:
                results[ext_col_name][feature] = "NA"

        lf_col_name = f"{base_model_prefix} {model_type_suffix} LF"
        total_possible_lf_selections = num_lf_seeds * MAX_SPLIT_COUNT_PER_SEED
        for feature, total_selections in feature_total_split_selections_lf.items():
            if total_possible_lf_selections > 0:
                percentage = (total_selections / total_possible_lf_selections) * 100
                results[lf_col_name][feature] = f"{percentage:.0f}%"
            else:
                results[lf_col_name][feature] = "NA"

    if os.path.exists(TWO_STEP_PATH):
        two_step_feature_totals = {}
        num_two_step_seeds = 0
        for seed_folder in os.listdir(TWO_STEP_PATH):
            seed_folder_full_path = os.path.join(TWO_STEP_PATH, seed_folder)
            if not os.path.isdir(seed_folder_full_path):
                continue

            results_path = os.path.join(seed_folder_full_path, "overall_results.json")
            if os.path.exists(results_path):
                try:
                    with open(results_path, "r") as f:
                        data = json.load(f)
                    if "selected_features_count" in data and data["selected_features_count"]:
                        num_two_step_seeds += 1
                        for feature, count_in_splits in data["selected_features_count"].items():
                            two_step_feature_totals[feature] = two_step_feature_totals.get(feature, 0) + count_in_splits
                except (json.JSONDecodeError, Exception):
                    pass

        two_step_col_name = "RF Regression Two-Step"  # New column name
        total_possible_two_step_selections = num_two_step_seeds * MAX_SPLIT_COUNT_PER_SEED
        results[two_step_col_name] = {}
        for feature, total_selections in two_step_feature_totals.items():
            if total_possible_two_step_selections > 0:
                percentage = (total_selections / total_possible_two_step_selections) * 100
                results[two_step_col_name][feature] = f"{percentage:.0f}%"
            else:
                results[two_step_col_name][feature] = "NA"

    return results


def create_comparison_table(results_data): # Removed min_overall_consistency_threshold parameter
    column_order = [
        "RF Binary EXT", "RF Binary LF",
        "XGB Binary EXT", "XGB Binary LF",
        "RF Regression EXT", "RF Regression LF",
        "XGB Regression EXT", "XGB Regression LF",
        "RF Regression Two-Step"
    ]

    all_features = set()
    for variant_data in results_data.values():
        all_features.update(variant_data.keys())

    # No filtering based on threshold. All features from all_features will be considered.
    features_to_include = sorted(list(all_features))


    # Prepare data for the new DataFrame
    new_table_rows_data = {}
    features_already_handled = set()

    # Process combined features
    for ext_feature, combo_info in FEATURE_COMBINATIONS.items():
        lf_feature = combo_info["lf_counterpart"]
        combined_name = combo_info["combined_name"]

        # Create combined row if *either* the EXT or LF part exists in the data
        # Since we removed the threshold, we just check for presence in all_features
        if ext_feature in features_to_include or lf_feature in features_to_include:
            row_data = {}
            for variant in column_order:
                if "EXT" in variant:
                    row_data[variant] = results_data.get(variant, {}).get(ext_feature, "")
                elif "LF" in variant:
                    row_data[variant] = results_data.get(variant, {}).get(lf_feature, "")
                else: # For the Two-Step column, use the LF feature if available, else EXT, else empty
                    if variant == "RF Regression Two-Step":
                        row_data[variant] = results_data.get(variant, {}).get(lf_feature, "") # Use LF for two-step
                    else:
                        row_data[variant] = results_data.get(variant, {}).get(ext_feature, "")

            new_table_rows_data[combined_name] = row_data
            features_already_handled.add(ext_feature)
            features_already_handled.add(lf_feature)

    # Add remaining features that are not part of any combination
    for feature in sorted(features_to_include):
        if feature not in features_already_handled:
            row_data = {}
            for variant in column_order:
                row_data[variant] = results_data.get(variant, {}).get(feature, "")
            new_table_rows_data[feature] = row_data

    # Create the new DataFrame from the prepared data
    final_feature_order_keys = list(new_table_rows_data.keys())

    def custom_sort_key(feature_name):
        is_combined = ' / ' in feature_name
        is_lf_only = feature_name.startswith('lf_') and not is_combined
        return (not is_combined, is_lf_only, feature_name)

    sorted_final_feature_order = sorted(final_feature_order_keys, key=custom_sort_key)

    table = pd.DataFrame.from_dict(new_table_rows_data, orient='index', columns=column_order)
    table = table.reindex(sorted_final_feature_order)

    return table.fillna("")


# Generate the table
results = load_model_results()

# Call create_comparison_table without the threshold parameter
comparison_table = create_comparison_table(results)

# Format display
pd.set_option('display.max_rows', None)
display(HTML("<h3>Feature Selection Consistency Across Models</h3>"))
display(HTML(comparison_table.to_html()))

,RF Binary EXT,RF Binary LF,XGB Binary EXT,XGB Binary LF,RF Regression EXT,RF Regression LF,XGB Regression EXT,XGB Regression LF,RF Regression Two-Step
during_total_rainfall / lf_during_max_6h_intensity,100%,100%,100%,,100%,100%,100%,16%,
post_max_6h / lf_post_max_24h,92%,100%,4%,52%,76%,100%,100%,100%,96%
pre_max_6h / lf_pre_max_6h,80%,100%,,8%,96%,100%,72%,100%,100%
dis_track_min,100%,100%,84%,100%,88%,100%,92%,100%,100%
vmax,100%,100%,100%,100%,100%,100%,100%,100%,100%
